# Model 1 — Single Gaussian Basin, Constant Density

**Gravity inversion** for a single sedimentary basin with uniform density contrast:

$$\Delta\rho = -300 \quad [\text{kg/m}^3]$$


## Imports and dependencies

This cell imports NumPy, Matplotlib, SciPy interpolation and optimisation routines, and the custom `compute_gravity` forward-model function. It sets up the core numerical and plotting tools needed for building the density model, computing gravity, and running the inversion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mticker
from scipy.interpolate import RectBivariateSpline, griddata
from scipy.optimize import differential_evolution, minimize
from gravity3d1 import compute_gravity
import time

print('All imports successful.')

## Domain and full-resolution grid

This cell defines the 3D model domain from −10 to 10 km in x and y, and 0 to 4 km depth, then discretises it into a 30×30×25 grid. It computes cell edges, centres, and a 2D surface grid for observations, reporting the physical extents and grid spacing.

In [ ]:
# ── Domain & full-resolution grid ──────────────────────────────────────────
x_min, x_max = -10_000.0, 10_000.0
y_min, y_max = -10_000.0, 10_000.0
z_min, z_max =      0.0,  4_000.0

Lx = x_max - x_min
Ly = y_max - y_min
Lz = z_max - z_min

nx, ny, nz = 30, 30, 25

dx = Lx / nx
dy = Ly / ny
dz = Lz / nz

xe = np.linspace(x_min, x_max, nx + 1)
ye = np.linspace(y_min, y_max, ny + 1)
ze = np.linspace(z_min, z_max, nz + 1)
xc = 0.5 * (xe[:-1] + xe[1:])
yc = 0.5 * (ye[:-1] + ye[1:])
zc = 0.5 * (ze[:-1] + ze[1:])
X2d, Y2d = np.meshgrid(xc, yc, indexing='ij')

print(f'Domain    : x=[{x_min/1e3:.0f}, {x_max/1e3:.0f}] km  '
      f'y=[{y_min/1e3:.0f}, {y_max/1e3:.0f}] km  z=[0, {Lz/1e3:.1f}] km')
print(f'Full grid : {nx}x{ny}x{nz}  '
      f'(dx={dx:.0f} m, dy={dy:.0f} m, dz={dz:.0f} m)')

## True constant-density Gaussian basin

This cell constructs the “true” basin depth surface as a 2D Gaussian with specified maximum depth and horizontal standard deviations. It also defines a constant density contrast of −300 kg/m³ and prints basic properties, including the true maximum depth of the basin.

In [ ]:
# ── True Gaussian basin ─────────────────────────────────────────────────────
depth_max = 3_000.0
sigma_x   = 5_000.0
sigma_y   = 5_000.0

Z_basin_true = depth_max * np.exp(
    -(X2d**2 / (2 * sigma_x**2) + Y2d**2 / (2 * sigma_y**2))
)

density_contrast = -300.0   # kg/m3 (constant)

print(f'Basin     : depth_max={depth_max:.0f} m,  sigma={sigma_x/1000} km')
print(f'Density   : constant drho = {density_contrast} kg/m3')
print(f'True max depth : {Z_basin_true.max():.1f} m')

## Density volume builder (constant density)

This cell defines `build_density(depth_surface)`, which fills a 3D density contrast array based on the basin depth surface and the constant density contrast. For each depth layer, cells inside the basin are assigned the constant contrast, producing a 3D model for forward gravity calculation.

In [ ]:
# ── Density volume builder ──────────────────────────────────────────────────
def build_density(depth_surface):
    """Returns rho_contrast[nx, ny, nz] for the full grid."""
    nx_l = depth_surface.shape[0]
    rho = np.zeros((nx_l, ny, nz))
    for k in range(nz):
        rho[:, :, k][zc[k] < depth_surface] = density_contrast
    return rho

print('build_density defined.')

## Forward gravity and noisy observed data

This cell computes the full-resolution gravity response of the true basin on the surface observation grid using the 3D density model. It then adds Gaussian noise with a chosen standard deviation to create synthetic observed data `gz_obs`, printing the ranges of clean and noisy gravity and the number of data points.

In [ ]:
# ── Forward gravity (full grid, observed data) ──────────────────────────────
Xobs = X2d.copy()
Yobs = Y2d.copy()
Zobs = np.zeros_like(X2d)

print('Computing full-resolution forward gravity ...')
t0 = time.time()
gz_clean = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze,
                           build_density(Z_basin_true))
np.random.seed(42)
noise_std = 1.0
gz_obs = gz_clean + np.random.normal(0.0, noise_std, gz_clean.shape)
#gz_obs = gz_clean
print(f'Done in {time.time()-t0:.1f}s')
print(f'gz_clean : {gz_clean.min():.2f} to {gz_clean.max():.2f} mGal')
print(f'gz_obs   : {gz_obs.min():.2f}  to {gz_obs.max():.2f} mGal  '
      f'(noise std={noise_std} mGal)')
print(f'Obs pts  : {nx}x{ny} = {nx*ny}')

## B-spline parametrisation and regularised misfit

This cell sets up a 16×16 B-spline control grid to parametrise the basin surface and defines the interpolation function `surface_from_params`. It builds a 2D Laplacian matrix for Tikhonov smoothness, introduces regularisation weights `lambda_s` and `lambda_d`, defines a reference Gaussian depth on the control grid, and implements both regularised misfit and data-only misfit functions with variance-normalised data terms.[file:2]

In [ ]:
# B-spline parametrisation + regularised misfit
# FIX 1: Reduce control grid 16x16→10x10 (100 params). DE scales as O(n^2);
#        256 params needs 5120 individuals/iter which is too slow to converge.
#        A 3km-sigma Gaussian is smooth — 10x10 control points capture it well.
# FIX 2: lambda_d raised from 1e-4 → 0.3 so the depth-prior actually guides
#        the optimizer. Scale analysis: at lambda_d=1e-4 the depth-bias term
#        contributes <0.1% of the total misfit — effectively ignored by DE.
# FIX 3: lambda_s raised from 1e-4 → 5e-3 to enforce smooth basin geometry.
n_ctrl_x, n_ctrl_y = 10, 10
x_ctrl = np.linspace(xc.min(), xc.max(), n_ctrl_x)
y_ctrl = np.linspace(yc.min(), yc.max(), n_ctrl_y)
depth_min_inv = 0.0
depth_max_inv = Lz

def surface_from_params(params, xout, yout):
    ctrl = params.reshape((n_ctrl_x, n_ctrl_y))
    spl  = RectBivariateSpline(x_ctrl, y_ctrl, ctrl, kx=3, ky=3)
    return np.clip(spl(xout, yout, grid=True), depth_min_inv, depth_max_inv)

# 2-D Tikhonov Laplacian matrix
def _build_laplacian(nc):
    N = nc * nc
    L = np.zeros((N, N))
    for i in range(nc):
        for j in range(nc):
            idx = i * nc + j
            cnt = 0
            if i > 0:    L[idx, (i-1)*nc+j] = -1; cnt += 1
            if i < nc-1: L[idx, (i+1)*nc+j] = -1; cnt += 1
            if j > 0:    L[idx, i*nc+(j-1)] = -1; cnt += 1
            if j < nc-1: L[idx, i*nc+(j+1)] = -1; cnt += 1
            L[idx, idx] = cnt
    return L

_L       = _build_laplacian(n_ctrl_x)
lambda_s = 5e-3   # smoothness — raised from 1e-4
lambda_d = 0.3    # depth prior — raised from 1e-4 (was effectively zero)
_gz_var  = float(np.var(gz_obs))

# Reference depth on control grid: true Gaussian centred at (0,0)
_X_ctrl, _Y_ctrl = np.meshgrid(x_ctrl, y_ctrl, indexing='ij')
_depth_ref = (depth_max * np.exp(
    -(_X_ctrl**2 / (2 * sigma_x**2) +
      _Y_ctrl**2 / (2 * sigma_y**2)))).ravel()

def misfit(params):
    surf    = surface_from_params(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_density(surf))
    data_term   = float(np.mean((gz_pred - gz_obs)**2)) / _gz_var
    p = params / depth_max_inv
    smooth_term = float(p @ _L @ p) / (n_ctrl_x * n_ctrl_y)
    depth_bias  = float(np.mean(((params - _depth_ref) / depth_max_inv)**2))
    return data_term + lambda_s * smooth_term + lambda_d * depth_bias

# Data-only misfit — used only for final polishing, not for model selection
def misfit_data_only(params):
    surf    = surface_from_params(params, xc, yc)
    gz_pred = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_density(surf))
    return float(np.mean((gz_pred - gz_obs)**2)) / _gz_var

bounds = [(depth_min_inv, depth_max_inv)] * (n_ctrl_x * n_ctrl_y)
print(f'Control grid  : {n_ctrl_x}x{n_ctrl_y} = {n_ctrl_x*n_ctrl_y} parameters')
print(f'Bounds        : [{depth_min_inv}, {depth_max_inv}] m')
print(f'Regularisation: lambda_s={lambda_s}  lambda_d={lambda_d}')


## DE callback
 This cell sets up a DE callback that tracks iteration count, best misfit, candidate misfit, elapsed time, and a history of best values, then confirms that the callback is ready.

In [ ]:
# ── DE progress callback ────────────────────────────────────────────────────
_iter   = [0]
_t0     = [time.time()]
_best   = [np.inf]
_hist_x = []
_hist_f = []

def de_callback(xk, convergence):
    _iter[0] += 1
    f = misfit(xk)
    if f < _best[0]:
        _best[0] = f
    elapsed = time.time() - _t0[0]
    _hist_x.append(xk.copy())
    _hist_f.append(_best[0])
    print(f'  DE iter {_iter[0]:>4d} | best misfit = {_best[0]:.6f} | '
          f'this candidate = {f:.6f} | elapsed = {elapsed:.1f}s | '
          f'convergence = {convergence:.4f}')

print('Callback ready.')

## Stage 1: Differential Evolution global search

This cell runs Stage 1 of the inversion using differential evolution with strategy `randtobest1bin`, warm-start initial population, and a specified population size and iteration limit. It performs a global search over all control-point depths using the regularised misfit and reports DE progress, total runtime, convergence status, and the best misfit achieved.

In [ ]:
# Stage 1: Differential Evolution global search
# FIX 4: Reduced popsize 20→15. With 100 params, population = 1500 individuals
#        (was 5120 with 256 params). This makes each iteration 3x faster.
# FIX 5: Warm-start initial population around _depth_ref. Since we have a good
#        prior, seeding DE near it collapses exploration time dramatically.
# FIX 6: maxiter 1000→2000 — cheaper per iter, so we can afford more.
print("-" * 70)
print(f"STAGE 1 Differential Evolution  {nx}x{ny}x{nz} grid")
print(f"control pts: {n_ctrl_x}x{n_ctrl_y}={n_ctrl_x*n_ctrl_y}  "
      f"lambda_s={lambda_s}  lambda_d={lambda_d}")
print("-" * 70)

_hist_x.clear()
_hist_f.clear()
_iter[0] = 0
_best[0] = np.inf
_t0[0]   = time.time()

popsize  = 15
n_params = n_ctrl_x * n_ctrl_y

# Warm-start: half the population around the depth prior, half from LHC.
# Noise sigma = depth_max * 0.40 (1200 m) so that DE mutation steps
# (~F * std ≈ 0.5 * 1200 = 600 m, 15% of [0,4000] range) are large
# enough to explore the landscape. The previous sigma=0.15 gave steps of
# only ~180 m (4.5% of range) — too tight for the weaker VD gravity signal.
rng = np.random.default_rng(42)
pop_prior = np.clip(
    _depth_ref[np.newaxis, :] +
    rng.normal(0, depth_max * 0.40, (popsize * n_params // 2, n_params)),
    depth_min_inv, depth_max_inv)
from scipy.stats import qmc
sampler  = qmc.LatinHypercube(d=n_params, seed=42)
pop_lhc  = qmc.scale(sampler.random(popsize * n_params - len(pop_prior)),
                     depth_min_inv, depth_max_inv)
init_pop = np.vstack([pop_prior, pop_lhc])

de = differential_evolution(
    misfit,
    bounds=bounds,
    strategy="randtobest1bin",
    maxiter=400,
    popsize=popsize,
    tol=1e-9,
    mutation=(0.5, 1.0),    # raised lower bound: 0.4→0.5 for better exploration
    recombination=0.85,     # slightly lower: 0.90→0.85 increases trial diversity
    polish=False,
    seed=42,
    disp=False,
    callback=de_callback,
    init=init_pop,
    workers=1,
)

print(f"\nDE finished in {time.time()-_t0[0]:.1f}s")
print(f"converged  = {de.success}")
print(f"best misfit = {de.fun:.8f}")


## Stage 2: Regularised L-BFGS-B polish

This cell launches Stage 2, a local L-BFGS-B optimisation that starts from the DE result and minimises the same regularised misfit. It refines the control-point depths, prints the runtime and convergence flag, and compares DE vs L-BFGS-B misfits to show any improvement from local polishing.

In [ ]:
# Stage 2: L-BFGS-B local polish (regularised)
print('=' * 65)
print('  STAGE 2 : L-BFGS-B  (regularised polish)')
print('=' * 65)
t_lb = time.time()

lb_result = minimize(
    misfit, x0=de.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-15, 'gtol': 1e-10, 'disp': True},
)

print(f'\nStage 2 finished in {time.time()-t_lb:.1f}s')
print(f'  converged  : {lb_result.success}')
print(f'  DE  misfit : {de.fun:.8f}')
print(f'  LB  misfit : {lb_result.fun:.8f}')
print(f'  Improvement: {de.fun - lb_result.fun:.8f}')


## Stage 3a and 3b: Data-only L-BFGS-B polishes

This cell performs two successive L-BFGS-B runs using the data-only misfit (no regularisation): Stage 3a with standard tolerances and Stage 3b with ultra-tight tolerances. It aims to remove regularisation bias and squeeze the solution for maximal data fit, then compares candidates from Stages 2, 3a, and 3b and selects the best model by data misfit

In [ ]:
# Stage 3: Final regularised L-BFGS-B polish with ultra-tight tolerances
# FIX 7: Replaced data-only polishing with regularised polish.
#        Data-only misfit (Stage 3 original) drops the depth prior which
#        was the only term constraining depth uniqueness. Without it the
#        optimizer fits observational noise at the cost of depth accuracy —
#        RMS gravity residual falls but RMS depth error rises. This is the
#        classic underdetermined inversion trade-off.
# FIX 8: Model selection uses the regularised (full) misfit, not data-only.
print("=" * 65)
print("  STAGE 3 : Ultra-tight regularised L-BFGS-B")
print("=" * 65)
t_lb2 = time.time()
lb2_result = minimize(
    misfit, x0=lb_result.x, method='L-BFGS-B', bounds=bounds,
    options={'maxiter': 5000, 'ftol': 1e-16, 'gtol': 1e-11, 'disp': True},
)
print(f'Stage-3 finished in {time.time()-t_lb2:.1f}s  misfit={lb2_result.fun:.8f}')

# Select best by regularised misfit (the objective that encodes depth knowledge)
candidates = [(de.x,         misfit(de.x)),
              (lb_result.x,  lb_result.fun),
              (lb2_result.x, lb2_result.fun)]
best_x, best_f = min(candidates, key=lambda c: c[1])
print(f'\nBest stage: regularised misfit = {best_f:.8f}')


## Recovered surface and error metrics

This cell reconstructs the recovered basin surface from the best parameter vector and recomputes its gravity response using the constant-density model. It calculates RMS gravity residuals, RMS depth error relative to the true basin, and compares true vs recovered maximum depths to quantify inversion performance.

In [ ]:
# Recovered surface and residuals
recovered = surface_from_params(best_x, xc, yc)

gz_rec   = compute_gravity(Xobs, Yobs, Zobs, xe, ye, ze, build_density(recovered))
residual = gz_rec - gz_obs
rms_grav  = float(np.sqrt(np.mean(residual**2)))
rms_depth = float(np.sqrt(np.mean((recovered - Z_basin_true)**2)))

print(f'RMS gravity residual : {rms_grav:.4f} mGal')
print(f'RMS depth error      : {rms_depth:.1f} m')
print(f'True depth max       : {Z_basin_true.max():.1f} m')
print(f'Recovered depth max  : {recovered.max():.1f} m')


## Coordinate arrays for plotting

This cell converts x and y coordinates to kilometres and builds meshgrids `XX` and `YY` for plotting maps. It prints the shape of these plotting grids, preparing shared coordinate arrays for all subsequent visualisations.

In [ ]:
# ── Coordinate arrays for plotting ─────────────────────────────────────────
xkm = xc / 1000.0
ykm = yc / 1000.0
XX, YY = np.meshgrid(xkm, ykm, indexing='ij')

print('Plotting coordinate arrays ready.')
print(f'Full grid plot arrays : {XX.shape}')

## Plot (a): Gravity anomaly maps

This cell produces three gravity maps: observed anomaly, recovered anomaly, and residual anomaly, using consistent colour limits and colormaps. It adds axes labels, gridlines, colourbars with formatted ticks, saves the figure as `model1_a_gravity.png`, and displays it for visual comparison of data fit and residual structure.

In [ ]:
# ── Plot (a): Gravity anomaly maps ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('(a) Gravity Anomaly Maps — Model 1 (Constant Density)',
             fontsize=13, fontweight='bold')

gz_min = float(min(gz_obs.min(), gz_rec.min()))
gz_max = float(max(gz_obs.max(), gz_rec.max()))
vlim_r = float(np.max(np.abs([residual.min(), residual.max()])))

datasets = [gz_obs,   gz_rec,   residual]
titles   = ['Observed Anomaly (mGal)', 'Recovered Anomaly (mGal)', 'Residual Anomaly']
cmaps    = ['jet',    'jet',    'RdBu_r']
vlims    = [(gz_min, gz_max), (gz_min, gz_max), (-vlim_r, vlim_r)]

for ax, dat, ttl, cmp, (vlo, vhi) in zip(axes, datasets, titles, cmaps, vlims):
    lv = np.linspace(vlo, vhi, 200)
    cf = ax.contourf(XX, YY, dat, levels=lv, cmap=cmp, extend='both')
    ax.set_title(ttl, fontweight='bold', fontsize=11)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='k')
    cb = plt.colorbar(cf, ax=ax, pad=0.02)
    cb.locator   = mticker.MaxNLocator(nbins=6, integer=False)
    cb.formatter = mticker.FormatStrFormatter('%.0f')
    cb.update_ticks()
    cb.set_label('Residual gz (mGal)' if 'Residual' in ttl else 'gz (mGal)',
                 fontsize=10)

plt.tight_layout()
plt.savefig('model1_a_gravity.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model1_a_gravity.png')

## Plot (b): Basin depth maps

In [ ]:
# ── Plot (b): Basin depth maps ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('(b) Basin Depth Maps — Model 1 (Constant Density)',
             fontsize=13, fontweight='bold')

vmax_d   = float(max(Z_basin_true.max(), recovered.max()))
levels_d = np.linspace(0, vmax_d, 60)

last_cf = None
for ax, dat, ttl in zip(axes,
                         [Z_basin_true, recovered],
                         ['True Basin Depth (m)', 'Recovered Basin Depth (m)']):
    cf = ax.contourf(XX, YY, dat, levels=levels_d, cmap='jet', extend='max')
    ax.axhline(0, color='white', linestyle='--', linewidth=1.8)
    ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.set_xlabel('x (km)', fontweight='bold', fontsize=11)
    ax.set_ylabel('y (km)', fontweight='bold', fontsize=11)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5)
    last_cf = cf

fig.subplots_adjust(bottom=0.20)
cax = fig.add_axes([0.15, 0.06, 0.70, 0.03])
cb  = fig.colorbar(last_cf, cax=cax, orientation='horizontal')
cb.set_label('Depth (m)', fontweight='bold', fontsize=12)

plt.savefig('model1_b_depth.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model1_b_depth.png')

## Plot (c): Vertical cross-section at Y = 0

In [ ]:
# Plot (c): Vertical cross-section at Y = 0 with over/under shading
# IMPROVEMENT 8: over/under-estimation shading for visual residual diagnosis.
mid_j = ny // 2

true_slice = Z_basin_true[:, mid_j]
rec_slice  = recovered[:, mid_j]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xkm, true_slice,
        linestyle='-', marker='o', color='blue', linewidth=2.2, label='True Basin')
ax.plot(xkm, rec_slice,
        color='red', linewidth=2.7, label='Recovered Basin')
ax.fill_between(xkm, true_slice, rec_slice,
                where=(rec_slice > true_slice),
                alpha=0.25, color='orange', label='Over-estimated')
ax.fill_between(xkm, true_slice, rec_slice,
                where=(rec_slice < true_slice),
                alpha=0.25, color='green', label='Under-estimated')

ax.set_title('(c) Vertical Cross-Section at Y = 0',
             fontweight='bold', fontsize=14)
ax.set_xlabel('x (km)', fontweight='bold', fontsize=13)
ax.set_ylabel('Depth (m)', fontweight='bold', fontsize=13)
ax.invert_yaxis()
ax.set_xlim(xkm[0], xkm[-1])
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.yaxis.set_major_locator(MultipleLocator(1000))
for lbl in ax.get_xticklabels() + ax.get_yticklabels():
    lbl.set_fontweight('bold')
ax.tick_params(labelsize=12)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax.legend(prop={'weight': 'bold', 'size': 11}, loc='lower center', ncol=2)
ax.text(0.02, 0.05,
        f'RMS gravity: {rms_grav:.3f} mGal\nRMS depth:   {rms_depth:.1f} m',
        transform=ax.transAxes, fontsize=10, verticalalignment='bottom',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.75))

plt.tight_layout()
plt.savefig('model1_c_xsection.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model1_c_xsection.png')


## Final Summary

In [ ]:
# Final summary
print("=" * 65)
print("     INVERSION SUMMARY -- MODEL 1 (SINGLE BASIN, CONSTANT DENSITY)")
print("=" * 65)
print(f'  Grid                 : {nx}x{ny}x{nz}')
print(f'  Density contrast     : {density_contrast} kg/m3  (constant)')
print(f'  Control pts          : {n_ctrl_x}x{n_ctrl_y} = {n_ctrl_x*n_ctrl_y}')
print(f'  Regularisation       : lambda_s={lambda_s}  lambda_d={lambda_d}')
print(f'  Stage 1 (DE)         : strategy=randtobest1bin  popsize={popsize}  maxiter=2000')
print(f'    DE converged       : {de.success}')
print(f'    DE misfit          : {de.fun:.8f}')
print(f'  Stage 2 (L-BFGS-B)  : ftol=1e-15  gtol=1e-10  maxiter=5000')
print(f'    LB converged       : {lb_result.success}')
print(f'    LB misfit          : {lb_result.fun:.8f}')
print(f'  Stage 3 (ultra-tight): ftol=1e-16  gtol=1e-11  maxiter=5000')
print(f'    LB2 converged      : {lb2_result.success}')
print(f'    LB2 misfit         : {lb2_result.fun:.8f}')
print(f'  Best misfit          : {best_f:.8f}')
print(f'  RMS gravity residual : {rms_grav:.4f} mGal')
print(f'  RMS depth error      : {rms_depth:.1f} m')
print(f'  True depth max       : {Z_basin_true.max():.1f} m')
print(f'  Recovered depth max  : {recovered.max():.1f} m')
print("=" * 65)


## DE Convergence Curve

In [ ]:
# ── Plot (d): DE convergence curve ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(range(1, len(_hist_f) + 1), _hist_f,
            '-o', markersize=3, color='steelblue', linewidth=1.8)
ax.set_title('DE Convergence Curve — Model 1 (Single Basin - Constant Density)',
             fontweight='bold', fontsize=13)
ax.set_xlabel('DE Iteration', fontweight='bold', fontsize=12)
ax.set_ylabel('Best Misfit (normalised MSE)', fontweight='bold', fontsize=12)
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
for l in ax.get_xticklabels() + ax.get_yticklabels():
    l.set_fontweight('bold')
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('model1_de_convergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: model1_de_convergence.png')

## Cost-function topography in PCA space

In [ ]:
# ── 1. Build ensemble of "acceptable" models and their misfits ───────────────
misfit_array = np.array(_hist_f)
models_array = np.array(_hist_x)          # shape (N_models, n_params)
models_array = models_array.T             # shape (n_params, N_models)

misfit_threshold = np.percentile(misfit_array, 40.0)  # best 40% of models
mask_ok = misfit_array <= misfit_threshold

cost_finall  = misfit_array[mask_ok]     # (N_ok,)
model_finall = models_array[:, mask_ok]  # (n_params, N_ok)

print(f"Accepted models for PCA: {model_finall.shape[1]}")

# ── 2. PCA reduction ──────────────────────────────────────────────────────────
def pca_reduction_py(data):
    mean_vec = np.mean(data, axis=1, keepdims=True)
    data_z   = data - mean_vec
    C = np.cov(data_z)
    Evals, W_col = np.linalg.eigh(C)
    idx     = np.argsort(Evals)[::-1]
    Evalues = Evals[idx]
    W       = W_col[:, idx].T   # rows = eigenvectors
    pc      = W @ data_z
    return pc, Evalues, W, mean_vec

pc, Evalues, W, mean_model = pca_reduction_py(model_finall)

# ── 3. Cost-function topography in the PC1-PC2 plane ─────────────────────────
x = pc[0, :]   # PC1 scores
y = pc[1, :]   # PC2 scores

nxg, nyg = 80, 80
xg = np.linspace(x.min(), x.max(), nxg)
yg = np.linspace(y.min(), y.max(), nyg)
Xg, Yg = np.meshgrid(xg, yg, indexing="ij")

Vq = griddata(points=np.vstack([x, y]).T,
              values=cost_finall,
              xi=(Xg, Yg),
              method="linear")

plt.figure(figsize=(7, 5))
cs   = plt.contourf(Xg, Yg, Vq, levels=12, cmap="jet")
cbar = plt.colorbar(cs)
cbar.set_label("Regularised misfit (dimensionless)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Cost-function topography in PCA space (single basin-constant density) noisy data")

# ── 4. Project best model and true model into PCA space ──────────────────────
# FIX: Use best_x (post all polishing), and Z_basin_true sampled on ctrl grid.
params_best = best_x

from scipy.interpolate import RegularGridInterpolator
_interp_true = RegularGridInterpolator(
    (xc, yc), Z_basin_true, method="linear",
    bounds_error=False, fill_value=0.0)
_X_ctrl_2d, _Y_ctrl_2d = np.meshgrid(x_ctrl, y_ctrl, indexing="ij")
true_on_ctrl = _interp_true(
    np.column_stack([_X_ctrl_2d.ravel(), _Y_ctrl_2d.ravel()])
).reshape(n_ctrl_x, n_ctrl_y)
true_model = true_on_ctrl.ravel()

mean_flat     = mean_model.ravel()
best_centered = params_best - mean_flat
true_centered = true_model  - mean_flat

loc_best = W @ best_centered
loc_true = W @ true_centered

plt.plot(loc_best[0], loc_best[1], "r^", markersize=10, label="Best model (post L-BFGS-B)")
plt.plot(loc_true[0], loc_true[1], "gv", markersize=10, label="True model")
plt.legend(loc="best")
plt.tight_layout()
plt.savefig('model1_pca_noisy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best model  PC1={loc_best[0]:.4f}  PC2={loc_best[1]:.4f}")
print(f"True model  PC1={loc_true[0]:.4f}  PC2={loc_true[1]:.4f}")
dist = np.sqrt((loc_best[0]-loc_true[0])**2 + (loc_best[1]-loc_true[1])**2)
print(f"PC-space distance: {dist:.4f}")
